# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [ ]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# Lists to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None,  # Default to None (indicating missing content)
        "error_code": None  # New field for failed downloads
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        file_entry["error_code"] = "Invalid URL"
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        file_entry["error_code"] = response.status_code  # Store specific HTTP error code
        print(f"❌ Failed to fetch {direct_url} - HTTP {response.status_code}: {http_err}")
        failed_downloads.append(file_entry)
    except requests.exceptions.RequestException as e:
        file_entry["error_code"] = "Request Error"
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")

# Read the Training Data

In [7]:
import pandas as pd
import numpy as np

In [8]:
#Converted to long format in R
df_train_raw = pd.read_csv("../data/annotations_long.csv", index_col="row_id")
keep = ["file_hash","type","subtype_confidence","notes_free_text","label"]
df_train = df_train_raw[keep].copy()

In [9]:
df_train

,file_hash,type,subtype_confidence,notes_free_text,label
row_id,,,,,
1,f8be35d0c20d1b1f3de4c44323e1780ee24f06893b6364...,I Na,3 - Highly confident,NaN,i_na_persistent
2,e97ca8a7f9734805832e5ae75442d19d3d7796b9a24190...,I K,2 - Mildly confident,STATES n l taul = 0.26*(v+50)/qtl,i_k_a_type_slow
4,606423f8f4a4f406f3387c7ee6f142bd121237272c347d...,Receptor,3 - Highly confident,NaN,r_gaba_type_a
5,99713c0032634e96cc7cde2dce02d8aa2baf75ad4cc835...,Receptor,3 - Highly confident,NaN,r_gaba_general
6,2d34ae241d628e7f25374c1bd84a8138cedc6a54689568...,Receptor,3 - Highly confident,mix,r_glutamate_ampa
...,...,...,...,...,...
462,ed87e94223dec25e43020f389207c0e362361efd05f3b0...,I Na,3 - Highly confident,NaN,i_na_transient
463,414840eea6fe6455d3377f99f43543155d2e4771b8cf8c...,I Ca,3 - Highly confident,NaN,i_ca_t_type_lt
464,43cdc726cbc3bc20eb08263de0028f8bfba35edba720eb...,Exclude,3 - Highly confident,NaN,exclude_accumulation_mechanism


# Read JSON file

In [1]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
pd.set_option("display.max_columns", None)

In [2]:
def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [161]:
# Function to extract mod directory from the URL
def get_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def get_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def get_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to extract all TITLE occurrences from .mod content
def get_title(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"^TITLE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all COMMENT sections from .mod content
def get_comment(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"COMMENT\s+(.*?)(?:\s+ENDCOMMENT|\Z)", content, re.DOTALL)  
    return matches if matches else None

# Function to extract all SUFFIX occurrences from .mod content
def get_suffix(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"SUFFIX\s+([^\n:\s]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None


def get_use_ion(content):
    """
    Extracts the ion names used in the 'USEION' statements from NEURON mod file content.

    Parameters:
    - content (str): The content of the .mod file.

    Returns:
    - list: A list of ions used in 'USEION' statements, or None if none are found.
    """
    if pd.isna(content):  
        return None
    
    # Find all occurrences of USEION followed by an ion name
    matches = re.findall(r"USEION\s+(\w+)", content, re.MULTILINE)

    return matches if matches else None


# Function to extract all ions listed after READ but stopping before WRITE, USEION, RANGE, GLOBAL, NONSPECIFIC_CURRENT, or VALENCE
def get_read_ion(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?=\s+(?:WRITE|USEION|RANGE|GLOBAL|NONSPECIFIC_CURRENT|VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return read_ions if read_ions else None  


# Function to extract all ions listed after WRITE, stopping before VALENCE
def get_write_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"WRITE\s+([^\n:]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  


def write_current_yn(ions):
    """
    Checks if mod_write_ion contains an ion that starts with 'i' (indicating a current).

    Args:
        write_ions (list): List of ions written in the mod file.

    Returns:
        int: 1 if any ion starts with 'i', otherwise 0.
    """
    if not ions:  # Handle empty lists or None
        return 0

    return int(any(ion.startswith("i") for ion in ions))


# Function to extract all NONSPECIFIC currents
def get_nonspecific_current(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n:]*)", content)

    if not matches:
        return None

    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  

#todo: should we assume we only want active variables or also extract ones that are commented out?
# Function to extract RANGE variables based on mode
def get_range(content, mode="active"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")


# Function to extract only active RANGE variables, stopping at colons and the end of the line
def get_range(content):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract all RANGE statements (each line separately), stopping before colons
    matches = re.findall(r"^\s*RANGE\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process active RANGE variables, ensuring they don't capture anything past the colon
    active_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return active_vars if active_vars else None  # Return only active variables
    
# Function to extract parameter names and values as a dictionary
def get_parameter(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"PARAMETER\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    param_dict = {}
    
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore commented-out lines
                continue
            param_match = re.match(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
            if param_match:
                param_name, param_value = param_match.groups()
                param_dict[param_name] = float(param_value)  

    return param_dict if param_dict else None  

# Function to extract only active STATE variables, ignoring comments (`:`) and unit values `(mV)`, etc.
def get_state(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    state_vars = []
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore fully commented-out lines
                continue
            line = re.split(r"\s*:\s*", line)[0]  # Remove inline comments (anything after `:`)
            clean_line = re.sub(r"\([^)]*\)", "", line).strip()  # Remove unit values
            if clean_line:
                state_vars.append(clean_line)

    return state_vars if state_vars else None  


# Function to extract only active GLOBAL variables, ignoring commented-out (`:`) ones
def get_global(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"^\s*GLOBAL\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    global_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return global_vars if global_vars else None  


def get_net_receive(content):
    """
    Extracts all NET_RECEIVE block arguments from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted NET_RECEIVE arguments, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of NET_RECEIVE and extract arguments
    matches = re.findall(r"^\s*NET_RECEIVE\s*\(\s*([\w, ]+)\s*\)", content, re.MULTILINE)

    if not matches:
        return None

    net_receive_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return net_receive_vars if net_receive_vars else None


def get_include(content):
    """
    Extracts the filename in the INCLUDE statement from MOD file content, ignoring comments.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted INCLUDE filenames, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of INCLUDE, ensuring commented-out ones (with ':') are ignored
    matches = re.findall(r"^\s*INCLUDE\s+\"([^\"]+)\"", content, re.MULTILINE)

    return matches if matches else None


def get_point_process(content):
    """
    Extracts the POINT_PROCESS name from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        str or None: The extracted POINT_PROCESS name, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Extract the POINT_PROCESS name, ignoring comments
    match = re.search(r"^\s*POINT_PROCESS\s+([^\n:]+)", content, re.MULTILINE)

    return match.group(1).strip() if match else None


    
# Function to extract webpage heading
def get_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def get_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def get_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def get_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found




def has_x86(url):
    """
    Checks if the given URL contains 'x86_64'.

    Parameters:
    - url (str): The URL to check.

    Returns:
    - int: 1 if 'x86_64' is present in the URL, 0 otherwise.
    """
    if pd.isna(url):  
        return None  # Return None for missing URLs
    return 1 if "x86_64" in url else 0


def has_error(error_code):
    """
    Checks if the given error code is not NaN (i.e., an error is present).

    Parameters:
    - error_code: The error code to check.

    Returns:
    - int: 1 if an error code is present (not NaN), 0 otherwise.
    """
    return 1 if error_code is not None else 0

def has_electrode_or_clamp(mod_name, content):
    """
    Checks whether 'clamp' is present in the mod file name OR 
    'ELECTRODE_CURRENT' is present in the NEURON mod file content.

    Parameters:
    - mod_name (str): The name of the .mod file.
    - content (str): The content of the .mod file.

    Returns:
    - int: 1 if either 'clamp' is in the mod name OR 'ELECTRODE_CURRENT' is in the content, 0 otherwise.
    """
    if pd.isna(mod_name) and pd.isna(content):
        return None  # Return None if both are missing

    has_clamp = bool(re.search(r"clamp", str(mod_name), re.IGNORECASE)) if pd.notna(mod_name) else False
    has_electrode = bool(re.search(r"\bELECTRODE_CURRENT\b", str(content))) if pd.notna(content) else False

    return 1 if has_clamp or has_electrode else 0

def map_ion(value):
    value = value.lower()  # Normalize to lowercase

    # Define regex-based categorization rules
    patterns = [
        (r'ca.*i$', 'ca_i'),
        (r'ca.*o$', 'ca_o'),
        (r'cl.*i$', 'cl_i'),
        (r'cl.*o$', 'cl_o'),
        (r'k.*i$', 'k_i'),
        (r'k.*o$', 'k_o'),
        (r'na.*i$', 'na_i'),
        (r'na.*o$', 'na_o'),
        (r'hco3.*i$', 'other_i'),
        (r'hco3.*o$', 'other_o'),
        (r'mgi$', 'mg_i'),  
        (r'mgo$', 'mg_o'),  
        (r'^img$', 'i_mg'),  
        (r'^emg$', 'e_mg'),
        (r'^e.*ca', 'e_ca'),
        (r'^e.*k', 'e_k'),
        (r'^e.*na', 'e_na'),
        (r'^e.*mg', 'e_mg'),
        (r'^e.*', 'e_other'),
        (r'^i.*ca', 'i_cal'),
        (r'^i.*k', 'i_k'),
        (r'^i.*cl', 'i_cl'),
        (r'^i.*na$', 'i_na'),  # FIX: Ensure "ina" is classified as "i_na"
        (r'^i.*mg', 'i_mg'),
        (r'^i.*', 'i_other'),
        (r'.*i$', 'other_i'),  # General rule: Anything ending in "i" is "other_i"
        (r'.*o$', 'other_o')   # General rule: Anything ending in "o" is "other_o"
    ]
    # Apply the regex patterns
    for pattern, category in patterns:
        if re.search(pattern, value):
            return category

    return "unknown"  # Default category if no match is found

def count_states(df, column_name="state"):
    """Counts the number of states in each row of the specified column."""
    df["count_states"] = df[column_name].apply(lambda x: len(x) if isinstance(x, list) else 0)
    return df

In [11]:
# Load JSON into DataFrame
df_all = pd.read_json("../data/mod_files.json")
df_all.set_index("row_id", inplace=True)
# Add exclusion criteria as flags
df_all["exclude_error"] = df_all["error_code"].apply(has_error)
df_all["exclude_x86"] = df_all["url"].apply(has_x86)
df_all['dataset'] = df_all['file_hash'].isin(df_train['file_hash']).map({True: 'train', False: 'test'})

In [12]:
df_train2 = df_train.merge(df_all, on=["row_id","file_hash"], how="left")

In [ ]:
#df_train2.loc[df_train2["exclude_x86"]==1].index == df_train2.loc[df_train2["label"]=="exclude_old_architecture"].index

In [13]:
df_train2["file_hash"].nunique()

407

In [15]:
df_train3 = df_train2.loc[(df_train2["exclude_error"] != 1) & (df_train2["exclude_x86"] != 1)] 

In [16]:
df_train3["file_hash"].nunique()

383

In [17]:
#All the "exclude" labels need to be collapsed to "neither"
#df_train4["label"].value_counts()
df_train3[df_train3["type"]=="Exclude"].index

Index([  7,  12,  15,  25,  37,  57,  63,  64,  74,  76,  82,  85, 107, 111,
       119, 122, 124, 129, 138, 147, 156, 167, 173, 175, 181, 192, 201, 202,
       205, 209, 215, 226, 235, 242, 244, 246, 260, 272, 277, 286, 290, 292,
       297, 302, 303, 309, 313, 315, 316, 323, 333, 338, 339, 345, 346, 348,
       350, 351, 354, 361, 362, 381, 384, 386, 388, 390, 393, 396, 398, 401,
       407, 432, 450, 461, 464],
      dtype='int64', name='row_id')

In [156]:
df_train4 = df_train3.copy()
df_train4['label'] = np.where(df_train4['type'] == "Exclude", 'neither', df_train4['label'])

In [21]:
#df_train4[df_train4["label"]=="neither"].index == df_train3[df_train3["type"]=="Exclude"].index

In [164]:
df = df_all.copy()
df["mod_name"] = df["url"].apply(get_fname)
df["has_electrode_or_clamp"] = df.apply(lambda row: has_electrode_or_clamp(row["mod_name"], row["content"]), axis=1)

In [165]:
df["has_electrode_or_clamp"]==1

,file_hash,raw_sha,count,url,download_url,content,error_code,exclude_error,exclude_x86,dataset,mod_name,has_electrode_or_clamp
row_id,,,,,,,,,,,,
1,f8be35d0c20d1b1f3de4c44323e1780ee24f06893b6364...,67b75245b9690ce219f34a5905425672dcb5c0661a885f...,6,https://modeldb.science/183300?tab=2&file=Shor...,https://modeldb.science/getModelFile?model=183...,TITLE Sodium persistent current\n\nCOMMENT\n ...,None,0,0,train,NaP,0
2,e97ca8a7f9734805832e5ae75442d19d3d7796b9a24190...,b45c13164f1eeaf4ba61ee8522304f7382fa0541831b1a...,1,https://modeldb.science/223649?tab=2&file=Altu...,https://modeldb.science/getModelFile?model=223...,TITLE K-A channel from Klee Ficker and Heinema...,None,0,0,train,kaprox,0
3,b428f4a84f4302dd5e9139beb678039621e4ee5e88b780...,b720bf29848dde82785dd101105f185ebcf9affa081dfd...,1,https://modeldb.science/3511?tab=2&file=mcn1/m...,https://modeldb.science/getModelFile?model=351...,INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}\n\nNEU...,None,0,0,test,mcn1_lg,0
4,606423f8f4a4f406f3387c7ee6f142bd121237272c347d...,81645a02d4dbf324cf8b82e8f31bda2ce46c3753f7a8ac...,1,https://modeldb.science/262115?tab=2&file=demo...,https://modeldb.science/getModelFile?model=262...,TITLE multiple GABAa receptors\n\nCOMMENT\n---...,None,0,0,train,multiGABAa,0
5,99713c0032634e96cc7cde2dce02d8aa2baf75ad4cc835...,5fe32f050754f534d60c7c126ecd805216089cd0594062...,1,https://modeldb.science/150288?tab=2&file=KimE...,https://modeldb.science/getModelFile?model=150...,:Interneuron Cells to Pyramidal Cells GABA wit...,None,0,0,train,interV2pyrDDANE_STFD,0
...,...,...,...,...,...,...,...,...,...,...,...,...
5129,79b0488211bde2819176d272a3baf1ea494f7a15b602a7...,7d66f936b9aff684c32bf25234f39c9e755d0fcda38a57...,9,https://modeldb.science/65218?tab=2&file=nrntr...,https://modeldb.science/getModelFile?model=652...,None,403,1,0,test,kdr_fs,0
5130,d8008fe9dc4f90a8bb5484995e07190bda986f48d36bd2...,309c68434d3809f6fd2d547a9869f202800320cc6e7a52...,1,https://modeldb.science/266989?tab=2&file=VMTh...,https://modeldb.science/getModelFile?model=266...,TITLE Hippocampal HH channels\n: \n:\n: Fast N...,None,0,0,test,TC_HH,0
5131,026e11a7203d24881dcf93b0c7c7fb7d15e6b7dd3d03d4...,8f8e9c80af3b9d715be639e6fa40670269332ae9bc264e...,1,https://modeldb.science/194897?tab=2&file=Dura...,https://modeldb.science/getModelFile?model=194...,": $Id: nsloc.mod,v 1.7 2013/06/20 salvad $\n:...",None,0,0,test,nsloc,0


In [157]:
# Extract features from url
df["heading"] = df["url"].apply(get_heading)  # Extract heading from webpage
df["citation"] = df["heading"].apply(get_citation)
df["first_author"] = df["citation"].apply(get_author)
df["year"] = df["citation"].apply(get_year)  # Now handles multiple years

df["dir"] = df["url"].apply(get_dir)
df["mod_name"] = df["url"].apply(get_fname)
df["model_id"] = df["url"].apply(get_model_id)

#  Extract features from content
df["title"] = df["content"].apply(get_title)
df["comment"] = df["content"].apply(get_comment)
df["suffix"] = df["content"].apply(get_suffix)
df["use_ion"] = df["content"].apply(get_use_ion)
df["read_ion"] = df["content"].apply(get_read_ion)
df["write_ion"] = df["content"].apply(get_write_ion)
df["range"] = df["content"].apply(get_range)
df["global"] = df["content"].apply(get_global)
df["parameter"] = df["content"].apply(get_parameter)
df["state"] = df["content"].apply(get_state)
df["net_receive"] = df["content"].apply(get_net_receive)
df["point_process"] = df["content"].apply(get_point_process)


# Exclusion criteria
df["has_electrode_or_clamp"] = df.apply(lambda row: has_electrode_or_clamp(row["mod_name"], row["content"]), axis=1)
#Binary Indicators
df["nonspecific_current"] = df["content"].apply(get_nonspecific_current)


In [128]:
import pandas as pd

def count_states(df, column_name="state"):
    """Counts the number of states in each row of the specified column."""
    df["count_states"] = df[column_name].apply(lambda x: len(x) if isinstance(x, list) else 0)
    return df["count_states"]


In [129]:
#todo: should we collapse low-frequency labels
df["label"].value_counts()

label
neither                                                 75
i_k_delayed_rectifier                                   25
r_glutamate_ampa                                        23
i_na_transient                                          22
i_ca_l_type_ht                                          20
r_glutamate_nmda                                        18
i_other_mixed                                           15
i_ca_t_type_lt                                          14
i_h_hyperpolarization_activated                         14
i_k_a_type_transient                                    13
i_na_persistent                                         13
i_k_ca_activated_general_bk_ik_sk_and_i_ahp_currents    12
i_k_m_type                                              11
i_na_general                                            10
r_gaba_type_a                                           10
i_k_a_type_slow                                         10
i_nonspecific                                     

In [68]:
# Flatten and collect unique ions from all three columns
unique_ions = set(df["write_ion"].dropna().explode()).union(
    set(df["read_ion"].dropna().explode()))

In [138]:
df["mod_name"].unique()

array(['NaP', 'kaprox', 'multiGABAa', 'interV2pyrDDANE_STFD',
       'pyrD2pyrVDA_STFD', 'MyExp2Syn', 'leakdepol', 'naxn', 'brain',
       'ca_a1h', 'ipulse3', 'km', 'GABAar', 'ProbAMPANMDA2group',
       'na1216_mut', 'CaHVA', 'KM', 'Interneuron_I_Na', 'fh', 'myexpsyn',
       'cat', 'na', 'h', 'nav1p9', 'Na', 'IL3', 'kd', 'hha_old',
       'chan_CaT', 'Ca', 'xtra', 'na3', 'ih', 'iCaL', 'nethhwbm', 'Kv4s',
       'naf97', 'ampa_kin', 'Purk_lkg', 'G_kaprox', 'mhh_Gfluct', 'kleck',
       'kapin', 'feature_weaver', 'CaT3_1_DP', 'ikdrd', 'BK_Slow',
       'Ca_HVA_hay', 'stellat_phasic', 'na_ion_dynamics', 'Cacum', 'hh2',
       'IM_cortex', 'hh3', 'ProbGABAAB_EMS', 'ChR2H134R', 'updown',
       'fastK', 'cad2', 'ar', 'newkca', 'kaf_ms', 'SynExp4NMDA2', 'FRAP',
       'cat_mig', 'infot', 'cf', 'nas', 'K_pyr_bk', 'naz_nature',
       'Gfluct', 'Cortical_Soma_I_K', 'n12_stoch', 'cal', 'na3rp',
       'pkdrd', 'cal2', 'pkdrs', 'ka', 'naps', 'kv72wt73wt', 'ic', 'cdp5',
       'ihpkj_cn', 'kcc

In [137]:
 set(df["suffix"].dropna().explode())

{'B_A',
 'B_Na',
 'CAn',
 'Ca',
 'CaBK',
 'CaDiffuse3',
 'CaHVA',
 'CaIntraCellDyn',
 'CaT',
 'CaT3_1_DP',
 'Ca_HVA_hay',
 'Ca_LVAst',
 'Cacum',
 'Cadynam',
 'Calcium',
 'Cav3_2',
 'ChR2',
 'ExIAF',
 'FRAP',
 'GRANULE_KCA',
 'GRC_KIR',
 'Golgi_CALC_ca2',
 'Golgi_KA',
 'Golgi_KM',
 'Golgi_lkg',
 'GrC_Kir',
 'Gran_CaPool_98',
 'HH',
 'HHk',
 'ICan',
 'INaP',
 'Icapn',
 'Ih',
 'Ikt1m4h',
 'K2',
 'K23cvode',
 'KA_bsg_yka',
 'KAsm',
 'KDRs',
 'KI',
 'KIn',
 'KM',
 'KPyr',
 'KaProx',
 'Kca',
 'Kdend',
 'Kdr1',
 'Kdrsd',
 'Km',
 'Kv42b',
 'Kv4s',
 'Kv7M',
 'Kx',
 'L_Ca',
 'Leak',
 'Llva',
 'NMDAKIT',
 'Na',
 'NaL',
 'NaP',
 'NaTs2_t',
 'Na_rat_ms',
 'Naaxon',
 'Nadend',
 'NaxSH0_ChannelML',
 'PasS',
 'Purk_lkg',
 'SK',
 'abBK',
 'ar',
 'axnode75',
 'bks',
 'body',
 'borgka',
 'brain',
 'ca',
 'caL',
 'cab',
 'cad',
 'cad2',
 'cadad',
 'cadifus',
 'cadyn',
 'cah',
 'cal',
 'calH',
 'caldyn',
 'calhh',
 'capr',
 'caq',
 'car',
 'cat',
 'cat1h',
 'cdp5',
 'ch_KCaS',
 'ch_Kdrp',
 'cldifus',
 'cld

In [115]:
View(df[df["read_ion2"].apply(lambda x: isinstance(x, list) and bool(set(x) & set(["unknown"])))]["read_ion"])


row_id
1350    [cai, POINTER, gbar]
Name: read_ion, dtype: object

In [152]:
View(df_train_raw[df_train_raw["label"]=="exclude_clamp"])                              

,file_hash,raw_sha,count,url,mod_file,type,subtype_confidence,ask_robert,annotated,notes_free_text,subtype,label
row_id,,,,,,,,,,,,
15,42c2d00c6e0e6a38f863fe7046a7664c8b33bfb31392f9c5473187182222a684,2030ca115cee73d030752189aec7a2cbd91dc2f9c502a40574df3775522d2da1,3,https://modeldb.science/144376?tab=2&file=Skolnik_python_WinogradEtAl2008/ipulse3.mod,Hodgkin-Huxley model of persistent activity in PFC neurons,Exclude,3 - Highly confident,n,y,NaN,subtype_1,exclude_clamp
37,1eb9f62f592115967bb9d3c1e99679339273c0eafa89bda09af1dcfae7c81037,17c820718a0d8f1fd853125d7d84645b225fd3b1ec97c71b2f622cb7f0515e66,1,https://modeldb.science/187604?tab=2&file=modeldbca1/xtra.mod,"Hippocampal CA1 NN with spontaneous theta, gamma: full scale & network clamp (Bezaire et al 2016)",Exclude,3 - Highly confident,n,y,NaN,subtype_1,exclude_clamp
129,57b2ba7ef6db8a60e427783c857fc42fc3ec216b1d767fdf1297e9c77d28360a,a9d8b5346680c3f586076438fc5361e7ea163ebe0c7839736b83b26a3832b25f,10,https://modeldb.science/184497?tab=2&file=RumbellEtAl2016/model/vsource.mod,"vsource.mod Patterned after svclmp.mod, a single electrode Voltage clamp mechanism.",Exclude,3 - Highly confident,n,y,v clamp,subtype_1,exclude_clamp
167,586121011312b02f59a6cdbfb20e9ac04a129aae31e5d15ddc988a5bb87321c1,c3bbbdec97786f712747eacc2266d1bd951453771202b5dca0856682c08c6613,1,https://modeldb.science/267293?tab=2&file=modulhcn/modulhcn_hay/GClampCa.mod,"Copied from IClamp, use amp and e instead of just amp",Exclude,3 - Highly confident,n,y,Iclamp,subtype_1,exclude_clamp
205,385a783b6e29491cf0bf64cdb4897aa264c679b77eeac72800a932622b3d50dc,a9c6e3bda7193f2febc8f3f84750aff09e31de3c1947e3cd45d9d73ee24304fd,2,https://modeldb.science/235769?tab=2&file=Kim2017/fig6/syn_ramp.mod,Point process for generation of ascending and descending variation in synaptic conductance over time.,Exclude,3 - Highly confident,n,y,NaN,subtype_1,exclude_clamp
215,aec23cd9b58c8b80eafae737724ab6303aa80ee6a780e0fc98138dc05d866b61,f5c3d45802dd0fa4d6cab7babf8dfc7df7a9b1919ef01d8e3e1a7afe765fa3af,1,https://modeldb.science/144589?tab=2&file=BerzThetaGamm2013/xtra.mod,Modulation of hippocampal rhythms by electric fields and network topology (Berzhanskaya et al. 2013),Exclude,3 - Highly confident,n,y,NaN,subtype_1,exclude_clamp
226,8261679b54d1539a74ba45619cc7ac3b1fa8de0148b85126a94af1a5d709f68e,b116f4a074ef2371eae5226812651b22652867513316bead5f1e12538dcf0b7c,1,https://modeldb.science/94321?tab=2&file=chain_1d/mechanisms/Isin.mod,"SIN current, Sinusoidal current model for membrane impedance analysis",Exclude,3 - Highly confident,n,y,NaN,subtype_1,exclude_clamp
277,5b3c88cead6f84a388d052ceeb188c70a58b2d7074f814490a1c63adbe13643b,1765ce8f8f3616a05be9369b7dc77ad79d8d0eed03901c317b85cf9c8d263f2d,2,https://modeldb.science/267293?tab=2&file=modulhcn/modulhcn_hay/GClamp.mod,"Copied from IClamp, use amp and e instead of just amp",Exclude,3 - Highly confident,n,y,IClamp,subtype_1,exclude_clamp
297,ba7e9a4ed60dbe3e479e3686ccf69d6c4fefbfdd4395f4b649f1ff09f7a34a20,f49ba4021ee79ff212ce63add526bb6d41715bd180ac3f19fdd862fe14d0ebde,4,https://modeldb.science/183300?tab=2&file=ShortEtAl2016/early_theta_version/order/order1.mod,Mitral cell activity gating by respiration and inhibition in an olfactory bulb NN (Short et al 2016),Exclude,3 - Highly confident,n,y,NaN,subtype_1,exclude_clamp


In [158]:
df[df["has_electrode_or_clamp"]==1]

,file_hash,type,subtype_confidence,notes_free_text,label,raw_sha,count,url,download_url,content,error_code,exclude_error,exclude_x86,dataset,heading,citation,first_author,year,dir,mod_name,model_id,title,comment,suffix,use_ion,read_ion,write_ion,range,global,parameter,state,net_receive,point_process,exclude_electrode,current,nonspecific_current,has_clamp,has_electrode_or_clamp
row_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
15,42c2d00c6e0e6a38f863fe7046a7664c8b33bfb31392f9...,Exclude,3 - Highly confident,NaN,neither,2030ca115cee73d030752189aec7a2cbd91dc2f9c502a4...,3,https://modeldb.science/144376?tab=2&file=Skol...,https://modeldb.science/getModelFile?model=144...,COMMENT\n ipulse3.mod\n Generates a train of...,None,0,0,train,Hodgkin-Huxley model of persistent activity in...,Winograd et al. 2008,Winograd,2008,Skolnik_python_WinogradEtAl2008,ipulse3,144376,None,[ipulse3.mod\n Generates a train of current p...,None,None,None,None,"[del, dur, per, num, amp, dc, i, pcount]",None,None,None,[w],Ipulse3,1,0,None,0,1
129,57b2ba7ef6db8a60e427783c857fc42fc3ec216b1d767f...,Exclude,3 - Highly confident,v clamp,neither,a9d8b5346680c3f586076438fc5361e7ea163ebe0c7839...,10,https://modeldb.science/184497?tab=2&file=Rumb...,https://modeldb.science/getModelFile?model=184...,TITLE vsource.mod\nCOMMENT\nPatterned after sv...,None,0,0,train,Rhesus Monkey Young and Aged L3 PFC Pyramidal ...,Rumbell et al. 2016,Rumbell,2016,None,vsource,184497,[vsource.mod],"[Patterned after svclmp.mod, a single electrod...",None,None,None,None,"[toff, amp, rs, vc, i]",None,{'rs': 1.0},None,None,Vsource,1,0,None,0,1
167,586121011312b02f59a6cdbfb20e9ac04a129aae31e5d1...,Exclude,3 - Highly confident,Iclamp,neither,c3bbbdec97786f712747eacc2266d1bd951453771202b5...,1,https://modeldb.science/267293?tab=2&file=modu...,https://modeldb.science/getModelFile?model=267...,"COMMENT\nCopied from IClamp, use amp and e ins...",None,0,0,train,Simulations of modulation of HCN channels in L...,"Mäki-Marttunen and Mäki-Marttunen, 2022",Mäki-Marttunen,2022,None,GClampCa,267293,None,"[Copied from IClamp, use amp and e instead of ...",None,[ca],[eca],None,"[del, dur, amp, ica]",None,None,None,None,GClampCa,0,0,None,1,1
226,8261679b54d1539a74ba45619cc7ac3b1fa8de0148b851...,Exclude,3 - Highly confident,NaN,neither,b116f4a074ef2371eae5226812651b22652867513316be...,1,https://modeldb.science/94321?tab=2&file=chain...,https://modeldb.science/getModelFile?model=943...,TITLE SIN current\r\n\r\nCOMMENT\r\n----------...,None,0,0,train,Inferring connection proximity in electrically...,Cali et al. 2007,Cali,2007,None,Isin,94321,[SIN current\r],[---------------------------------------------...,None,None,None,None,"[del, dur, amp, i, freq, off]",None,None,None,None,Isin,1,0,None,0,1
277,5b3c88cead6f84a388d052ceeb188c70a58b2d7074f814...,Exclude,3 - Highly confident,IClamp,neither,1765ce8f8f3616a05be9369b7dc77ad79d8d0eed03901c...,2,https://modeldb.science/267293?tab=2&file=modu...,https://modeldb.science/getModelFile?model=267...,"COMMENT\nCopied from IClamp, use amp and e ins...",None,0,0,train,Simulations of modulation of HCN channels in L...,"Mäki-Marttunen and Mäki-Marttunen, 2022",Mäki-Marttunen,2022,None,GClamp,267293,None,"[Copied from IClamp, use amp and e instead of ...",None,None,None,None,"[del, dur, amp, e, i]",None,None,None,None,GClamp,1,0,None,1,1
297,ba7e9a4ed60dbe3e479e3686ccf69d6c4fefbfdd4395f4...,Exclude,3 - Highly confident,NaN,neither,f49ba4021ee79ff212ce63add526bb6d41715bd180ac3f...,4,https://modeldb.science/183300?tab=2&file=Shor...,https://modeldb.science/getModelFile?model=183...,: demonstrate the order in which various tasks...,None,0,0,train,Mitral cell activity gating by respiration and...,Short et al 2016,Short,2016,None,order1,183300,None,None,[order1],None,None,None,[index],None,{'index': 0.0},None,None,None,1,0,[i],0,1
345,a1baff83bc033fbbfd90d7f0343d4097f01b9a75957e0a...,Exclude,3 - Highly confident,Clamp,neither,b959ee2ca9b2526083534c88d026adebea8cc123d93c2a...,1,https://

In [122]:
# Apply function with unique category enforcement
df["read_ion"] = df["content"].apply(get_read_ion)
df["write_ion"] = df["content"].apply(get_write_ion)
df['read_ion2'] = df['read_ion'].apply(lambda ion_list: list(set(map_ion(ion) for ion in ion_list)) if isinstance(ion_list, list) else None)

In [124]:
#View(df.loc[:,["read_ion","read_ion2"]])

In [ ]:
#METADATA
#model_id
#model_name
#author
#year

#ION-SPECIFIC FEATURES
#read_ion for major molecules (k, na, ca, cl, mg). if not major, then collapse into other
#write_ion for major molecules (k, na, ca, cl, mg), if not major, then collapse into other
#type of ion read for major molecules (outside concentration, inside concentration, reversal potential, current)
#type of ion written for major molecules (outside concentration, inside concentration, reversal potential, current)
#suffix: cad, nothing, kdr, na, kca
#min and max taus for each tau
#number of gates/states


#RECEPTOR-SPECIFIC FEATURES
#net_receive for major receptors. if not major then collapse into other
#point_process for major receptors. if not major then collapse into other

#NEITHER FEATURES
#has_electrode_current: has an ELECTRODE CURRENT in the mod file or a CLAMP in the mod file name 
#is_neither: does not read or write ions AND does not have a nonspecific current AND does not have a net receive 


In [ ]:
df


In [ ]:
View(df["suffix"].value_counts())

In [ ]:
View(df.loc[470,["url"]])

In [ ]:
#df[(df["mod_exclude"] == "not_found") | (df["mod_exclude"] == "x86_64")].shape

# Flagging Issues

In [ ]:
#row_id: 477 - commented out ranges included (https://modeldb.science/266818?tab=2&file=Ventricular_GUI.CircRes.ModelDB/Kss.mod)
#row_id: 481 - has comments with mod_state variables (https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod)
#row_id: 483 - has units in the mod_state ( https://modeldb.science/195666?tab=2&file=DewellGabbiani2018/mod_files/LGMD_KD_ca3.mod)
#row_id: 483 - was only extracting ONE parameter instead of multiple parameters (fixed)
#row_id 31 - has VALENCE in the write_ion (https://modeldb.science/116862?tab=2&file=b09jan13/IL3.mod)
#row_id 99 - need to fix use_ion

# Questions

In [ ]:
#Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?
#How to capture variables that are commented out vs. not? 
#what's the best way to capture 
"""example of a function that captures both active and commented out variables. I think its cleaner to capture active only?
import re
import pandas as pd

# Function to extract RANGE variables based on mode
def get_range(content, mode="all"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")

"""

In [ ]:
View(df.loc[481,["url","mod_state"]],50)

In [ ]:
import re





In [159]:
! git add .
! git commit -m "combined clamp and electrode"
! git push 

[main b11f42a] combined clamp and electrode
 1 file changed, 865 insertions(+), 214 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 64 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 6.61 KiB | 3.30 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To github.com:innacohen/mod-extract.git
   3d63d50..b11f42a  main -> main
